# Prepare AlphaGenome variant inputs

Extract enhancer-linked SNP–gene pairs from annotated Hi-C loops and add REF/ALT alleles from UCSC dbSNP155.

## 1. Load annotated loops

In [1]:
from pathlib import Path
import pandas as pd

# Input and output directories
BASE_DIR = Path("../data")
PROJECT_DIR = Path("../results")

# Input files
LOOPS_FILE = BASE_DIR / (
    "189_clean_genes_final_candidate_loops.regulatory_direction_fixed."
    "with_diffHiC_label_fuzzy30kb.bedpe"
)
UCSC_FILE = BASE_DIR / "SNP_input_alphagenome.tsv"

# Create output directory
PROJECT_DIR.mkdir(parents=True, exist_ok=True)

# Load annotated Hi-C loops
if not LOOPS_FILE.is_file():
    raise FileNotFoundError(LOOPS_FILE)

loops = pd.read_csv(LOOPS_FILE, sep="\t", dtype=str).fillna("")
print(f"Annotated loops: {len(loops):,}")

Annotated loops: 189


## 2. Extract SNP–gene pairs

In [2]:
BOOLEAN_COLUMNS = [
    "a1_has_snp", "a2_has_snp", "a1_has_enh", "a2_has_enh",
    "a1_has_tss", "a2_has_tss",
]
REQUIRED_COLUMNS = BOOLEAN_COLUMNS + [
    "loop_id", "chr1", "x1", "x2", "chr2", "y1", "y2",
    "anchor1_LD_snps", "anchor2_LD_snps",
    "anchor1_TSS_hits_10kb", "anchor2_TSS_hits_10kb",
    "source", "active_enhancer_anchor", "diff_HiC_label_fuzzy30kb",
]
# Check required columns
missing = sorted(set(REQUIRED_COLUMNS) - set(loops.columns))
if missing:
    raise ValueError(f"Missing expected loop columns: {missing}")

In [3]:
# Convert TRUE/FALSE labels to booleans
for col in BOOLEAN_COLUMNS:
    normalized = loops[col].str.strip().str.lower()
    unexpected = ~normalized.isin(["true", "false"])
    if unexpected.any():
        raise ValueError(f"Unexpected values in {col}: {normalized[unexpected].unique().tolist()}")
    loops[col] = normalized.eq("true")

# Find anchors containing both SNPs and active enhancers
loops["a1_snp_enh"] = loops["a1_has_snp"] & loops["a1_has_enh"]
loops["a2_snp_enh"] = loops["a2_has_snp"] & loops["a2_has_enh"]

def split_values(value):
    return [v.strip() for v in str(value).split(",") if v.strip()]

In [4]:
# Get SNP-gene pairs from loops
records = []
for _, loop in loops.iterrows():
    for snp_side, gene_side, eligible in [
        (1, 2, loop["a1_snp_enh"] and loop["a2_has_tss"]),
        (2, 1, loop["a2_snp_enh"] and loop["a1_has_tss"]),
    ]:
        if not eligible:
            continue
        snps = split_values(loop[f"anchor{snp_side}_LD_snps"])
        genes = split_values(loop[f"anchor{gene_side}_TSS_hits_10kb"])
        snp_chr = loop["chr1"] if snp_side == 1 else loop["chr2"]
        gene_chr = loop["chr1"] if gene_side == 1 else loop["chr2"]
        snp_start, snp_end = (loop["x1"], loop["x2"]) if snp_side == 1 else (loop["y1"], loop["y2"])
        gene_start, gene_end = (loop["x1"], loop["x2"]) if gene_side == 1 else (loop["y1"], loop["y2"])
        # Store SNP-gene pairs
        for rsid in snps:
            for gene in genes:
                records.append({
                    "loop_id": loop["loop_id"], "rsid": rsid,
                    "snp_chr": snp_chr, "snp_anchor": f"anchor{snp_side}",
                    "snp_anchor_start": int(snp_start), "snp_anchor_end": int(snp_end),
                    "target_gene": gene, "target_chr": gene_chr,
                    "target_anchor": f"anchor{gene_side}",
                    "target_anchor_start": int(gene_start), "target_anchor_end": int(gene_end),
                    "source": loop["source"],
                    "active_enhancer_anchor": loop["active_enhancer_anchor"],
                    "diff_HiC_label_fuzzy30kb": loop["diff_HiC_label_fuzzy30kb"],
                })

In [5]:
# Keep unique SNP–gene relationships
if not records:
    raise ValueError("No SNP–gene pairs matched the distal enhancer/TSS criteria")
pairs = pd.DataFrame.from_records(records).drop_duplicates().reset_index(drop=True)
# Summarize unique rsIDs and loops
unique_snps = pairs[["rsid", "snp_chr"]].drop_duplicates().sort_values(["snp_chr", "rsid"])
if unique_snps.groupby("rsid")["snp_chr"].nunique().gt(1).any():
    raise ValueError("Some rsIDs map to multiple chromosomes; inspect input annotations")
loop_summary = pairs.groupby("loop_id", as_index=False).agg(
    n_snps=("rsid", "nunique"), n_target_genes=("target_gene", "nunique"),
    snp_anchor=("snp_anchor", "first"), source=("source", "first"),
)

## 3. Save SNP and loop tables

In [6]:
# Save SNP and loop tables
pairs.to_csv(PROJECT_DIR / "139_distal_loop_SNP_gene_pairs.tsv", sep="\t", index=False)
unique_snps.to_csv(PROJECT_DIR / "139_loops_unique_rsids.tsv", sep="\t", index=False)
unique_snps["rsid"].drop_duplicates().to_csv(PROJECT_DIR / "rsids_for_UCSC_or_VEP.txt", header=False, index=False)
loop_summary.to_csv(PROJECT_DIR / "139_loop_summary.tsv", sep="\t", index=False)
# Check how many loops, SNPs, and genes were retained
print(f"Pairs: {len(pairs):,} | Loops: {pairs.loop_id.nunique():,} | rsIDs: {pairs.rsid.nunique():,} | Genes: {pairs.target_gene.nunique():,}")
pairs.head()

Pairs: 1,421 | Loops: 139 | rsIDs: 711 | Genes: 201


,loop_id,rsid,snp_chr,snp_anchor,snp_anchor_start,snp_anchor_end,target_gene,target_chr,target_anchor,target_anchor_start,target_anchor_end,source,active_enhancer_anchor,diff_HiC_label_fuzzy30kb
0,36,rs10864629,chr1,anchor2,6625000,6650000,PLEKHG5,chr1,anchor1,6475000,6500000,COPD,both,
1,36,rs10864629,chr1,anchor2,6625000,6650000,TNFRSF25,chr1,anchor1,6475000,6500000,COPD,both,
2,36,rs11892,chr1,anchor2,6625000,6650000,PLEKHG5,chr1,anchor1,6475000,6500000,COPD,both,
3,36,rs11892,chr1,anchor2,6625000,6650000,TNFRSF25,chr1,anchor1,6475000,6500000,COPD,both,
4,36,rs12079872,chr1,anchor2,6625000,6650000,PLEKHG5,chr1,anchor1,6475000,6500000,COPD,both,


## 4. Add UCSC REF/ALT alleles

The UCSC input is a dbSNP155 table. Convert the zero-based `chromStart` to a one-based position.

In [7]:
# Load UCSC dbSNP annotations
if not UCSC_FILE.is_file():
    raise FileNotFoundError(f"UCSC dbSNP155 table not found: {UCSC_FILE}")
ucsc = pd.read_csv(UCSC_FILE, sep="\t", dtype=str).rename(
    columns={"#chrom": "chr", "name": "rsid", "ref": "REF"}
)
required_ucsc = {"chr", "rsid", "REF", "chromStart", "class", "alts"}
missing = required_ucsc - set(ucsc.columns)
if missing:
    raise ValueError(f"Missing UCSC columns: {sorted(missing)}")

# Keep SNVs matching the Hi-C-linked rsIDs
ucsc = ucsc.loc[ucsc["rsid"].isin(unique_snps["rsid"]) & ucsc["class"].str.lower().eq("snv")].copy()

In [8]:
# Convert UCSC zero-based coordinates to one-based positions
ucsc["position"] = pd.to_numeric(ucsc["chromStart"], errors="raise").astype(int) + 1
# Split multiallelic records into alternate alleles
ucsc["ALT_list"] = ucsc["alts"].fillna("").map(split_values)
ucsc["n_ALT"] = ucsc["ALT_list"].map(len)
ucsc = ucsc.merge(unique_snps, on="rsid", how="left", validate="many_to_one")

In [10]:
# Check chromosome agreement between UCSC and SNP annotations
chrom_mismatch = ucsc[ucsc["chr"] != ucsc["snp_chr"]].copy()

print(f"Chromosome mismatches: {len(chrom_mismatch)}")

if not chrom_mismatch.empty:
    display(chrom_mismatch[["rsid", "chr", "snp_chr"]].drop_duplicates())

# Remove chromosome mismatches
ucsc = ucsc[ucsc["chr"] == ucsc["snp_chr"]].copy()

# Keep variants with a single-base reference allele
ucsc = ucsc[
    (ucsc["REF"].str.len() == 1) &
    (ucsc["n_ALT"] > 0)
].copy()

# Expand variants with multiple alternate alleles
variants = ucsc.explode("ALT_list")
variants = variants.rename(columns={
    "ALT_list": "ALT",
    "n_ALT": "n_ALT_for_rsid"
})

# Keep single-nucleotide substitutions
variants = variants[
    (variants["ALT"].str.len() == 1) &
    (variants["ALT"] != variants["REF"])
].copy()

# Select final columns and remove duplicates
variants = variants[
    ["rsid", "chr", "position", "REF", "ALT", "n_ALT_for_rsid"]
].drop_duplicates().reset_index(drop=True)

print(f"Final REF/ALT variants: {len(variants):,}")

Chromosome mismatches: 11


,rsid,chr,snp_chr
249,rs7248169,chr19_ML143376v1_fix,chr19
250,rs62122064,chr19_ML143376v1_fix,chr19
251,rs35421192,chr19_ML143376v1_fix,chr19
252,rs62122066,chr19_ML143376v1_fix,chr19
253,rs34597097,chr19_ML143376v1_fix,chr19
254,rs1531212,chr19_ML143376v1_fix,chr19
255,rs7250028,chr19_ML143376v1_fix,chr19
256,rs2594713,chr19_ML143376v1_fix,chr19
257,rs2594714,chr19_ML143376v1_fix,chr19
258,rs2607276,chr19_ML143376v1_fix,chr19


Final REF/ALT variants: 1,174


In [11]:
# Check missing SNPs
missing_rsids = sorted(set(unique_snps["rsid"]) - set(variants["rsid"]))
print(f"Expected rsIDs: {unique_snps.rsid.nunique():,}")
print(f"rsIDs with usable SNV alleles: {variants.rsid.nunique():,}")
print(f"Unique REF→ALT substitutions: {len(variants):,}")
print(f"rsIDs without usable SNV alleles: {len(missing_rsids):,}")
if missing_rsids:
    pd.Series(missing_rsids, name="rsid").to_csv(PROJECT_DIR / "rsids_missing_alleles.tsv", sep="\t", index=False)

variants.head()

Expected rsIDs: 711
rsIDs with usable SNV alleles: 710
Unique REF→ALT substitutions: 1,174
rsIDs without usable SNV alleles: 1


,rsid,chr,position,REF,ALT,n_ALT_for_rsid
0,rs7519565,chr1,19375162,T,A,3
1,rs7519565,chr1,19375162,T,C,3
2,rs7519565,chr1,19375162,T,G,3
3,rs72651487,chr1,19379595,G,A,1
4,rs17394154,chr1,19379891,A,C,2


## 5. Save AlphaGenome inputs

In [13]:
# Set output filenames
variant_file = PROJECT_DIR / "alphagenome_variant_input.clean.tsv"
relationship_file = PROJECT_DIR / "alphagenome_SNP_loop_gene_input.tsv"

# Merge variants with SNP-gene pairs
relationships = pairs.merge(
    variants, on="rsid", how="left", validate="many_to_many"
)
valid_pairs = relationships["REF"].notna() & relationships["ALT"].notna()
if (relationships.loc[valid_pairs, "snp_chr"] != relationships.loc[valid_pairs, "chr"]).any():
    raise ValueError("Chromosome mismatch after merging variant and Hi-C tables")

# Save variant and SNP–gene input tables
variants.to_csv(variant_file, sep="\t", index=False)
relationships.to_csv(relationship_file, sep="\t", index=False)
ucsc.loc[ucsc["n_ALT"].gt(1), ["rsid", "chr", "position", "REF", "alts", "n_ALT"]].to_csv(
    PROJECT_DIR / "alphagenome_multiallelic_variants.tsv", sep="\t", index=False
)
# Summarize the final outputs
print(f"Variant definitions: {len(variants):,}")
print(f"SNP–loop–gene rows: {len(relationships):,}")
print(f"Rows missing allele annotations: {(~valid_pairs).sum():,}")
# print(f"Saved: {variant_file}")
# print(f"Saved: {relationship_file}")

Variant definitions: 1,174
SNP–loop–gene rows: 2,295
Rows missing allele annotations: 1


### Output checks

Compare the number of loops, unique rsIDs, and REF/ALT substitutions against the original analysis before running predictions. The UCSC input and Hi-C loop annotations must use the same genome assembly.